# 📘 本章总结：卷积层里的多输入多输出通道

## 一、为什么需要多通道？
- **真实图像本就是多通道**：RGB 图像有 3 个颜色通道，医学影像、卫星图等更多。
- **单通道卷积核表达力有限**：一个核只能识别**一种**模式（如水平边缘）。
- **多通道**让网络在每层同时学习**多种特征**（边缘、纹理、颜色块……），并在深层组合成复杂模式。

---

## 二、多输入通道（$c_i > 1$）
### 1. 形状约定
| 张量 | 形状 |
|---|---|
| 输入 X | $(c_i,\, n_h,\, n_w)$ |
| 卷积核 K | $(c_i,\, k_h,\, k_w)$ ← **核的通道数 = 输入通道数** |
| 输出 Y | $(n_h - k_h + 1,\, n_w - k_w + 1)$ ← **单通道** |

### 2. 计算规则
对每个输入通道**分别**做 2D 互相关，再把所有通道的结果**逐元素相加**：
$$
Y[i,j] \;=\; \sum_{c=0}^{c_i - 1} \; \sum_{a,b} \; X[c,\, i+a,\, j+b] \cdot K[c,\, a,\, b]
$$

**🔤 符号语义说明：**
| 符号 | 含义 | 取值范围 |
|---|---|---|
| $c_i$ | **input channels** — 输入通道数（如 RGB 图像 $c_i=3$） | 正整数 |
| $c$ | 输入通道索引（遍历所有通道并求和） | $0 \le c < c_i$ |
| $i, j$ | 输出特征图上的**空间坐标**（行号、列号） | $0 \le i < n_h - k_h + 1$，$0 \le j < n_w - k_w + 1$ |
| $a, b$ | 卷积核内部的**局部偏移坐标**（核窗口内的行/列） | $0 \le a < k_h$，$0 \le b < k_w$ |
| $n_h, n_w$ | 输入图像的**高度**、**宽度** | 正整数 |
| $k_h, k_w$ | 卷积核的**高度**、**宽度**（如 3×3 卷积则 $k_h=k_w=3$） | 正奇数（常见） |
| $X[c, i{+}a, j{+}b]$ | 输入张量在第 $c$ 通道、位置 $(i+a, j+b)$ 的像素值 | 浮点数 |
| $K[c, a, b]$ | 卷积核在第 $c$ 通道、核内位置 $(a, b)$ 的权重 | 可学习参数 |
| $Y[i, j]$ | 输出特征图在位置 $(i, j)$ 的值（**所有输入通道之和**） | 浮点数 |

> 💡 关键理解：双重求和 $\sum_c \sum_{a,b}$ 同时遍历**通道维** and **空间核窗口**——通道维求和后通道信息"塌缩"，空间维求和实现局部加权。

对应代码（PyTorch 框架接口）：
```python
import torch.nn.functional as F
# 框架要求 4D 输入 (B, c_i, n_h, n_w)、4D 核 (c_o, c_i, k_h, k_w)
X4 = X.unsqueeze(0)   # (c_i,n_h,n_w) → (1,c_i,n_h,n_w)，补上 batch 维
K4 = K.unsqueeze(0)   # (c_i,k_h,k_w) → (1,c_i,k_h,k_w)，整理成 c_o=1 的单输出核
Y = F.conv2d(X4, K4)  # 输出 (1, 1, n_h-k_h+1, n_w-k_w+1)
```
**核心**：框架的 `F.conv2d` 内部已经把"按通道分别卷积再逐元素相加"实现好了——
我们只需把输入升到 4D、把核整理成 `(c_o, c_i, k_h, k_w)`，多输入通道的求和由框架自动完成。

---

## 三、多输出通道（$c_o > 1$）
### 1. 形状约定
| 张量 | 形状 |
|---|---|
| 输入 X | $(c_i,\, n_h,\, n_w)$ |
| 卷积核 K | $(c_o,\, c_i,\, k_h,\, k_w)$ ← **4D**！外层是输出通道维 |
| 输出 Y | $(c_o,\, n_h - k_h + 1,\, n_w - k_w + 1)$ |

### 2. 计算规则
**每个输出通道对应一个独立的 3D 核**，框架用一个 4D 核张量一次性表达全部 $c_o$ 个核：
```python
import torch.nn.functional as F
X4 = X.unsqueeze(0)   # (c_i,n_h,n_w) → (1,c_i,n_h,n_w)
# K 直接就是 4D：(c_o, c_i, k_h, k_w)，无需手动遍历堆叠
Y = F.conv2d(X4, K)   # 输出 (1, c_o, n_h-k_h+1, n_w-k_w+1)
```
- 框架核张量第 0 维就是输出通道维 $c_o$，**无需** `for k in K` 手动遍历；
- 多输出通道的"堆叠"由 `F.conv2d` 内部完成，输出第 1 维即 $c_o$，省去了 `torch.stack`。

### 3. 直观理解
- 每个输出通道 = 一个**模式检测器**（"竖线探测器"、"红色块探测器"……）；
- 网络越深，输出通道往往越多（如 ResNet：64 → 128 → 256 → 512），表示越抽象的特征组合。

---

## 四、$1 \times 1$ 卷积：通道维的"全连接"
### 1. 关键观察
当 $k_h = k_w = 1$ 时，卷积**不再混合空间邻域**，只对**同一空间位置**的所有输入通道做线性组合 → 等价于在通道维做一次**全连接层**。

### 2. 数学等价
把输入展平成 $(c_i, h \cdot w)$、核展平成 $(c_o, c_i)$，则：
$$
\underbrace{Y}_{c_o \times hw} \;=\; \underbrace{K}_{c_o \times c_i} \;\cdot\; \underbrace{X}_{c_i \times hw}
$$

**🔤 符号语义说明：**
| 符号 | 含义 | 形状 |
|---|---|---|
| $X$ | 输入特征图**展平后**的矩阵——每行是一个输入通道，每列是一个空间位置 | $c_i \times (h \cdot w)$ |
| $K$ | $1\times 1$ 卷积核**展平后**的矩阵——每行是一个输出通道的"通道混合权重" | $c_o \times c_i$ |
| $Y$ | 输出特征图**展平后**的矩阵——每行是一个输出通道，每列是一个空间位置 | $c_o \times (h \cdot w)$ |
| $c_i$ | input channels — 输入通道数 | 正整数 |
| $c_o$ | **output channels** — 输出通道数（= 1×1 卷积核个数） | 正整数 |
| $h, w$ | 特征图的**高度**、**宽度**（1×1 卷积不改变空间尺寸） | 正整数 |
| $h \cdot w$ | 空间位置总数（把 2D 平面拉成 1D 长度） | 正整数 |
| $\cdot$ | **矩阵乘法**（`torch.matmul`），不是逐元素积 | — |

> 💡 关键理解：这个公式揭示了 $1\times 1$ 卷积 = 一次**矩阵乘法** $K \cdot X$——每个空间位置独立地把 $c_i$ 维输入向量通过线性变换 $K$ 映射成 $c_o$ 维输出向量，本质上是**通道维度上的全连接层**。

用 PyTorch 框架接口实现就是一次普通卷积（`kernel_size=1`）：
```python
import torch.nn.functional as F
X4 = X.unsqueeze(0)   # (c_i,h,w) → (1,c_i,h,w)
# K 形状 (c_o, c_i, 1, 1)，即 1×1 卷积核
Y = F.conv2d(X4, K)   # 输出 (1, c_o, h, w)，空间尺寸不变
```
框架的 1×1 卷积内部正是"逐空间位置做 $K \cdot X$ 矩阵乘"，
与上面手写的 `torch.matmul(K, X)` 数学完全等价（误差 < $10^{-6}$，仅由浮点累加顺序导致）。

### 3. 工程价值
- **通道升维 / 降维**：用极小的参数量改变通道数（ResNet bottleneck、Inception）；
- **跨通道信息融合**：不引入空间感受野 of the 扩张；
- **几乎免费的非线性**：1×1 卷积后接 ReLU 即可。

---

## 五、二维卷积层完整形态
| 维度 | 含义 |
|---|---|
| 输入 X | $(B,\, c_i,\, n_h,\, n_w)$（B 为 batch） |
| 核 W | $(c_o,\, c_i,\, k_h,\, k_w)$ |
| 偏置 b | $(c_o,)$（每个输出通道一个标量） |
| 输出 Y | $(B,\, c_o,\, n_h',\, n_w')$ |

**🔤 符号语义说明：**
| 符号 | 含义 |
|---|---|
| $B$ | **batch size** — 一次前向传播中并行处理 of the 样本数 |
| $c_i$ | input channels — 输入通道数 |
| $c_o$ | output channels — 输出通道数（= 卷积核的数量） |
| $n_h, n_w$ | 输入特征图的高、宽 |
| $n_h', n_w'$ | 输出特征图的高、宽（受 padding/stride/kernel 影响） |
| $k_h, k_w$ | 卷积核的高、宽 |
| $W$ | 卷积核**权重张量**（4D，可学习） |
| $b$ | 偏置向量（每个输出通道共享一个标量偏置） |

**参数量公式：**
$$
\text{参数量} \;=\; \underbrace{c_o \cdot c_i \cdot k_h \cdot k_w}_{\text{权重 } W} \;+\; \underbrace{c_o}_{\text{偏置 } b}
$$

> 💡 关键理解：**参数量与输入空间尺寸 $n_h, n_w$ 完全无关**——这就是 CNN 相比 MLP 的核心优势：**参数共享 + 局部连接**。无论输入是 28×28 还是 224×224，同一层卷积的参数数量不变。

---

## 六、PyTorch API 关键细节
```python
nn.Conv2d(in_channels, out_channels, kernel_size,
          stride=1, padding=0, ...)
```
- ⚠️ **参数顺序是"先输入、后输出"**（cell-23 中的注释写反了，是常见笔误）；
- `nn.Conv2d` 期望 4D 输入 `(B, c_i, H, W)`，单图测试需手动 `reshape((1,1)+X.shape)` 升维；
- 内部权重形状即 $(c_o, c_i, k_h, k_w)$，可通过 `conv2d.weight.shape` 验证。

---

## 七、易错点与设计直觉
1. **核的通道数 ≠ 输出通道数**：核的"内层通道维"必须等于**输入**通道数 $c_i$；"外层维"才是**输出**通道数 $c_o$。
2. **多通道求和发生在通道维**：不要误以为每个通道单独输出——多输入通道 of the 2D 卷积只产出**一张** feature map。
3. **1×1 卷积不是无用操作**：虽然不混合空间信息，但在通道维度上是完全可学的线性变换，是现代 CNN 的关键构件。
4. **设计直觉**：
   - 浅层用大核（如 7×7）+ 少通道，提取低级特征；
   - 深层用小核（3×3 或 1×1）+ 多通道，组合高级语义；
   - 1×1 卷积常作为"通道适配器"穿插在网络中。

---

## 八、现实实战案例：ResNet 中的通道降维与升维（Bottleneck 模块）

在实际生产中，如何将**多输入输出通道**与 **$1\times 1$ 卷积**结合起来解决真正的工程问题？最经典的杰作就是 **ResNet（残差网络）中的 Bottleneck（瓶颈）模块**。

### 1. 痛点：深层网络的计算量爆炸
在深层网络中（如 ResNet-50），特征图的通道数非常多（例如 $c_i = 256$）。如果我们直接使用标准的 $3\times 3$ 卷积提取空间特征，且保持通道数不变（$c_o = 256$）：
- 权重参数量为：$256 \times 256 \times 3 \times 3 \approx 589,824$ 个参数。
- 当网络有几十层时，计算量 and 内存占用将极其恐怖。

### 2. 破局者：Bottleneck 的三步走策略
ResNet 巧妙地设计了 **“压缩-卷积-膨胀”** 的三步结构，利用 $1\times 1$ 卷积作为通道的“拉链”：

1. **通道降维（压缩）**：使用 **$1\times 1$ 卷积**将通道数从 $256$ 骤降到 $64$。
2. **空间特征提取（卷积）**：使用普通的 **$3\times 3$ 卷积**在低维通道数（$64$）上进行空间特征提取。
3. **通道升维（膨胀）**：再使用 **$1\times 1$ 卷积**将通道数从 $64$ 恢复回 $256$，以便与残差连接相加。

```
输入 X: 256通道  -->  [1x1 卷积: 降维至 64通道]  -->  [3x3 卷积: 提取特征]  -->  [1x1 卷积: 升维至 256通道]  --> 输出 Y
```

### 3. 参数量大pk：为什么能节省参数？
让我们来算一下 Bottleneck 设计的权重参数量：
- 第一步（$1\times 1$ 降维）：$256 \times 64 \times 1 \times 1 = 16,384$
- 第二步（$3\times 3$ 提取）：$64 \times 64 \times 3 \times 3 = 36,864$
- 第三步（$1\times 1$ 升维）：$64 \times 256 \times 1 \times 1 = 16,384$
- **总权重参数量**：$16,384 + 36,864 + 16,384 = \mathbf{69,632}$。

对比直接 $3\times 3$ 卷积的 **$589,824$**，参数量和计算量**缩减了将近 8.5 倍**！这就是 $1\times 1$ 卷积降维的强大工业价值。

### 4. 工业级实战代码（见下方可运行代码单元格）
我们在下方提供了一个完整的、可运行的 PyTorch 演示。您可以通过运行它，亲自验证多通道在 Bottleneck 中的流动规律，以及其惊人的参数压缩比！

In [5]:
# 🚀 知识点结晶：ResNet Bottleneck 现实实战案例与参数PK
import torch
from torch import nn

class ResNetBottleneck(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, out_channels):
        super().__init__()
        # 1. 1x1 卷积压缩通道 (降维)
        self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)
        
        # 2. 3x3 卷积提取空间特征 (用 padding=1 保持高宽不变)
        self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)
        
        # 3. 1x1 卷积膨胀通道 (升维)
        self.conv3 = nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x  # 残差捷径连接
        
        # 第一步：1x1 卷积降维
        out1 = self.conv1(x)
        out1 = self.bn1(out1)
        out1 = self.relu(out1)
        
        # 第二步：3x3 卷积提取特征
        out2 = self.conv2(out1)
        out2 = self.bn2(out2)
        out2 = self.relu(out2)
        
        # 第三步：1x1 卷积升维
        out3 = self.conv3(out2)
        out3 = self.bn3(out3)
        
        # 残差相加并激活
        out3 += identity
        out = self.relu(out3)
        
        # 返回输出以及中间过程的特征图形状，方便直观学习
        return out, (out1.shape, out2.shape, out3.shape)

# 1. 模拟现实生产输入 (例如：ResNet-50 中典型的特征图尺寸，256通道，56x56分辨率)
in_c, mid_c, out_c = 256, 64, 256
X_dummy = torch.randn(1, in_c, 56, 56)

# 2. 实例化 Bottleneck 模块并前向传播
bottleneck_block = ResNetBottleneck(in_c, mid_c, out_c)
Y_out, stage_shapes = bottleneck_block(X_dummy)
shape1, shape2, shape3 = stage_shapes

print("=== 🛠️ ResNet Bottleneck 各阶段特征图形状变化 ===")
print(f"输入特征图形状 X:       {list(X_dummy.shape)}")
print(f"第一步 (1x1 降维后):    {list(shape1)}  <-- 通道从 {in_c} 压缩到 {mid_c}")
print(f"第二步 (3x3 卷积后):    {list(shape2)}  <-- 空间特征提取完毕，保持通道")
print(f"第三步 (1x1 升维后):    {list(shape3)}  <-- 通道恢复为 {out_c}")
print(f"最终输出形状 Y:         {list(Y_out.shape)}  <-- 残差相加与 ReLU 激活后")

# 3. 参数量大对比 (直观展示 1x1 卷积在通道维“压缩-升维”所带来的巨大性能优势)
# 如果不使用 bottleneck，直接用标准的 3x3 卷积提取特征且保持通道数为 256
std_conv3x3 = nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False)

# 计算各自的权重参数量
params_std = sum(p.numel() for p in std_conv3x3.parameters())
params_bottleneck = sum(p.numel() for p in bottleneck_block.parameters() if p.requires_grad)

print("\n=== 📊 工业级参数效率大PK ===")
print(f"直接使用 3x3 卷积权重参数量:    {params_std:,} 个")
print(f"Bottleneck 三步走权重参数量:    {params_bottleneck:,} 个 (已包含 BN 归一化层参数)")
reduction_ratio = params_std / params_bottleneck
print(f"🔥 参数量缩减比例:             {reduction_ratio:.2f} 倍！")
print("=" * 45)

=== 🛠️ ResNet Bottleneck 各阶段特征图形状变化 ===
输入特征图形状 X:       [1, 256, 56, 56]
第一步 (1x1 降维后):    [1, 64, 56, 56]  <-- 通道从 256 压缩到 64
第二步 (3x3 卷积后):    [1, 64, 56, 56]  <-- 空间特征提取完毕，保持通道
第三步 (1x1 升维后):    [1, 256, 56, 56]  <-- 通道恢复为 256
最终输出形状 Y:         [1, 256, 56, 56]  <-- 残差相加与 ReLU 激活后

=== 📊 工业级参数效率大PK ===
直接使用 3x3 卷积权重参数量:    589,824 个
Bottleneck 三步走权重参数量:    70,400 个 (已包含 BN 归一化层参数)
🔥 参数量缩减比例:             8.38 倍！


### 📖 PyTorch 核心 API 使用介绍

在上述实战代码中，我们使用了 PyTorch 最核心的两个神经网络层：`nn.Conv2d` 和 `nn.BatchNorm2d`。为了帮助深入学习，以下是它们的核心参数、维度设计和工作机制详解。

---

#### 1. 二维卷积层：`torch.nn.Conv2d`

##### 🔹 核心构造参数
```python
nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True)
```
- **`in_channels`** (int): 输入通道数 $c_i$。例如彩色 RGB 图像为 3，本章实战中的降维输入为 256。
- **`out_channels`** (int): 输出通道数 $c_o$，即该层使用的**卷积核数量**。每一个通道对应特征图上一种特定的特征模式提取。
- **`kernel_size`** (int 或 tuple): 卷积核的空间大小。若是 $1\times1$ 卷积则传入 `1`（仅混合通道，不提取空间）；若是 $3\times3$ 卷积则传入 `3`。
- **`stride`** (int 或 tuple, 可选): 步幅，即核每次滑动的步长。默认为 1。
- **`padding`** (int 或 tuple, 可选): 填充。在边缘补零以控制输出高宽。本章实战中 $3\times3$ 卷积设置 `padding=1` 目的是**保持输出空间尺寸不变**。
- **`bias`** (bool, 可选): 是否启用可学习偏置。在接 BatchNorm 时，因为 BN 内部包含均值归一化与可学偏移，**偏置起不到任何作用**，通常显式设为 `bias=False` 以节约参数量和显存。

##### 🔹 维度流转与数学关系
- **期望输入形状**: 4D Tensor `(B, c_i, H_in, W_in)`
- **计算输出形状**: 4D Tensor `(B, c_o, H_out, W_out)`
  - 输出的高 $H_{out}$ 和宽 $W_{out}$ 分别为：
  $$
  H_{out} = \lfloor \frac{H_{in} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \rfloor + 1
  $$
- **可学习权重参数量形状**:
  - 权重 `weight.shape`: `(c_o, c_i, k_h, k_w)` ← **这也印证了前文卷积核必须是 4D 的原理！**
  - 偏置 `bias.shape`: `(c_o,)`（若 `bias=True`）

---

#### 2. 二维批量归一化层：`torch.nn.BatchNorm2d`

##### 🔹 工作机制说明
在深度网络中，深层神经元输入的分布会随前一层参数的变化而剧烈抖动（即内部协变量偏移 Internal Covariate Shift）。BatchNorm 通过对每个特征通道（通道维）进行跨 Batch、跨空间位置的均值和方差标准化，使数据保持稳定的分布，**极大加速网络收敛并防止梯度爆炸**。

其计算公式如下（对于通道 $c$ 中的所有元素）：
$$
\hat{x}_c = \frac{x_c - \mathrm{E}[x_c]}{\sqrt{\mathrm{Var}[x_c] + \epsilon}} \quad \Rightarrow \quad Y_c = \gamma_c \hat{x}_c + \beta_c
$$

##### 🔹 核心参数说明
```python
nn.BatchNorm2d(num_features, eps=1e-05, momentum=0.1, affine=True)
```
- **`num_features`** (int): 需要做归一化的**特征通道数**（通常直接传入前一层卷积的 `out_channels`）。
- **`eps`** (float, 可选): 一个极小值 $\epsilon$（默认 `1e-5`），加在方差分母上避免除零错。
- **`momentum`** (float, 可选): 动量因子，用于更新运行时的全局均值和全局方差。
- **`affine`** (bool, 可选): 仿射变换开关。若为 `True`（默认），则引入两个可学习的通道级别参数：拉伸因子 `weight` ($\gamma$) 和偏移因子 `bias` ($\beta$)。它们在网络反向传播中自动学习更新。

##### 🔹 关键状态变量 (学理要点) ⚠️
`BatchNorm2d` 包含两类性质完全不同的状态参数：
1. **可学习参数** (在 `state_dict()` 中，需要反向传播更新梯度)：
   - `weight` ($\gamma$): 形状为 `(num_features,)`，初始化为全 1。
   - `bias` ($\beta$): 形状为 `(num_features,)`，初始化为全 0。
2. **统计非学习变量** (在 `state_dict()` 中，通过前向传播的动量公式累计更新，无梯度)：
   - `running_mean` (全局累计均值): 形状为 `(num_features,)`。
   - `running_var` (全局累计方差): 形状为 `(num_features,)`。

##### 🔹 训练（Train）与测试（Eval）的行为差异
- **在训练状态下 (`model.train()`)**：BN 实时计算当前 Batch 数据的均值和方差做归一化，并利用动量不断更新全局 `running_mean` 和 `running_var`。
- **在评估状态下 (`model.eval()`)**：BN **停止计算 Batch 的均值方差**，而是直接套用训练阶段累计下来的全局 `running_mean` 和 `running_var` 进行标准化。这保证了模型推理时的确定性和一致性（单张图输入与多张图批量输入时，计算结果完全相同）。

# 1. 多个输入通道

以下是多输入通道互相关运算的计算示意图：

![conv-multi-in](https://zh-v2.d2l.ai/_images/conv-multi-in.svg)

In [ ]:
# 多输入通道互相关运算
import torch
from d2l import torch as d2l
from torch import nn

# 多通道输入运算
def corr2d_multi_in(X,K):
    return sum(d2l.corr2d(x,k) for x,k in zip(X,K)) # X,K为3通道矩阵，for使得对最外面通道进行遍历        

X = torch.tensor([[[0.0,1.0,2.0],[3.0,4.0,5.0],[6.0,7.0,8.0]],
                  [[1.0,2.0,3.0],[4.0,5.0,6.0],[7.0,8.0,9.0]]])
K = torch.tensor([[[0.0,1.0],[2.0,3.0]],[[1.0,2.0],[3.0,4.0]]])

print(X.shape)
print(K.shape)

mulit_in = corr2d_multi_in(X,K)
print("mulit_in",mulit_in.shape)

# 多输出通道运算
def corr2d_multi_in_out(X,K):  # X为3通道矩阵，K为4通道矩阵，最外面维为输出通道      
    return torch.stack([corr2d_multi_in(X,k) for k in K],0) # 大k中每个小k是一个3D的Tensor。0表示stack堆叠函数里面在0这个维度堆叠。           

K = torch.stack((K, K+1, K+2),0) # K与K+1之间的区别为K的每个元素加1
print(K.shape)
print(corr2d_multi_in_out(X,K))

## 📖 `d2l.corr2d` 函数介绍

### 1. 函数签名
```python
d2l.corr2d(X, K) → Tensor
```

### 2. 功能说明
计算**单通道二维互相关运算**（Cross-Correlation）——这是 CNN 的最基本操作：让卷积核 $K$ 在输入 $X$ 上滑动，每个位置做一次"逐元素乘 + 求和"。

> ⚠️ 严格来说叫"**互相关**"而非"卷积"：数学上的卷积要求把核翻转 180°，但深度学习里默认不翻转（反正核是学出来的，翻不翻转效果等价）。

### 3. 参数说明
| 参数 | 类型 | 形状 | 含义 |
|---|---|---|---|
| `X` | Tensor (2D) | $(n_h,\, n_w)$ | **单通道**输入矩阵 |
| `K` | Tensor (2D) | $(k_h,\, k_w)$ | **单通道**卷积核 |
| 返回 | Tensor (2D) | $(n_h - k_h + 1,\, n_w - k_w + 1)$ | 互相关输出特征图 |

### 4. 数学定义
$$
Y[i,j] \;=\; \sum_{a=0}^{k_h-1} \sum_{b=0}^{k_w-1} X[i+a,\, j+b] \cdot K[a,\, b]
$$

| 符号 | 含义 |
|---|---|
| $i, j$ | 输出特征图上的空间坐标 |
| $a, b$ | 卷积核窗口内的局部偏移 |
| $X[i+a, j+b]$ | 输入在感受野内对应位置的像素值 |
| $K[a, b]$ | 卷积核对应位置的权重 |

### 5. 等价实现（d2l 源码）
```python
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y
```
- 双层 `for` 遍历**输出**位置 $(i, j)$；
- `X[i:i+h, j:j+w]` 切出感受野（与核同形状）；
- `* K` 做**逐元素乘**，`.sum()` 求和得到一个标量。

### 6. 与 `torch.nn.functional.conv2d` 的区别
| 维度 | `d2l.corr2d` | `F.conv2d` |
|---|---|---|
| 输入形状 | 2D `(H, W)` | 4D `(B, C_in, H, W)` |
| 核形状 | 2D `(kH, kW)` | 4D `(C_out, C_in, kH, kW)` |
| 通道支持 | 仅单通道 | 多输入/多输出通道 |
| 用途 | **教学演示** | 生产环境（GPU 加速、自动微分） |

### 7. 在本章中的应用
`corr2d_multi_in` 把 `d2l.corr2d` 当作"单通道原子操作"反复调用，再把所有通道结果加起来，从而扩展到**多输入通道**：
```python
def corr2d_multi_in(X, K):
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))
```
- `zip(X, K)`：把 3D 输入 `X` 和 3D 核 `K` 沿**通道维**配对成 `(x, k)` 二元组；
- `d2l.corr2d(x, k)`：每对做一次**单通道**互相关，得到一张 2D 特征图；
- `sum(...)`：把 $c_i$ 张特征图**逐元素相加**，得到最终的单通道输出。

> 💡 关键理解：**多通道卷积 = 单通道卷积（`d2l.corr2d`）按通道求和**——理解了这一点，就理解了 CNN 多通道运算的全部本质。

## 📖 `torch.stack` 函数介绍

### 1. 函数签名
```python
torch.stack(tensors, dim=0, *, out=None) → Tensor
```

### 2. 功能说明
**沿一个新维度**将一组张量拼接起来。所有输入张量**必须形状相同**，输出张量的维度数为 **输入维度数 + 1**。

### 3. 参数说明
| 参数 | 类型 | 默认 | 含义 |
|---|---|---|---|
| `tensors` | Sequence[Tensor] | 必填 | 待拼接 of the 张量序列，**所有张量形状必须相同** |
| `dim` | int | `0` | **新插入**的维度位置，范围 `[0, N]`（N 为输入维度数） |
| `out` | Tensor | `None` | 可选的输出张量（性能优化用，一般不传） |

### 4. 与 `torch.cat` 的关键区别 ⚠️
| 函数 | 行为 | 输出维度数 |
|---|---|---|
| `torch.stack` | **新增**一个维度 | 输入维度 + 1 |
| `torch.cat` | 沿**已有**维度拼接 | 输入维度不变 |

例如：3 个形状为 `(2, 3)` 的张量
- `torch.stack(..., dim=0)` → `(3, 2, 3)` ← 新增维度 0
- `torch.cat(..., dim=0)` → `(6, 3)` ← 沿原维度 0 拼接

### 5. 典型用法示例
```python
x = torch.randn(2, 3)   # 形状 (2, 3)

# dim=0：新维度在最前面
torch.stack((x, x), dim=0).shape   # → (2, 2, 3)

# dim=1：新维度插入到中间
torch.stack((x, x), dim=1).shape   # → (2, 2, 3)

# dim=2 或 dim=-1：新维度在最后面
torch.stack((x, x), dim=-1).shape  # → (2, 3, 2)
```

### 6. 在本章中的应用
`corr2d_multi_in_out` 中用 `torch.stack(..., 0)` **沿新维度 0** 把 $c_o$ 个单通道卷积结果堆叠成 $(c_o, h, w)$：
```python
return torch.stack([corr2d_multi_in(X, k) for k in K], 0)
```
这里**必须用 `stack` 而不是 `cat`**——因为我们要**新增**一个输出通道维 $c_o$，而不是沿空间维拼接。

# 2. 多输入通道与多输出通道（使用框架）

在实际的深度学习项目中，我们不会自己实现 `corr2d_multi_in` 或 `corr2d_multi_in_out`，而是直接使用 PyTorch 提供的 `nn.Conv2d`（面向对象接口）或 `torch.nn.functional.conv2d`（函数式接口）。

它们内部集成了多输入通道与多输出通道的运算，且支持高性能 GPU 加速和自动梯度计算。

### 维度约定说明 ⚠️
- 框架输入 $X$ 期望是 4D 张量，形状为：`(batch_size, in_channels, height, width)`。
- 框架权重 $W$（卷积核）期望是 4D 张量，形状为：`(out_channels, in_channels, kernel_height, kernel_width)`。

为了使用框架，我们需要对前面自定义的 3D 输入 `X` 以及卷积核进行维度扩展（升维）。

In [ ]:
# 使用 PyTorch 框架验证多输入通道、多输出通道 of the 互相关计算
import torch.nn.functional as F

# 1. 扩展输入维度为 4D: (batch_size=1, in_channels=2, height=3, width=3)
X_4d = X.unsqueeze(0)

# 2. 验证：多输入单输出
# 我们从多输出卷积核 K 中提取第 0 个通道权重，并重构成 4D: (out_channels=1, in_channels=2, kH=2, kW=2)
K_single_out_4d = K[0].unsqueeze(0) if K.ndim == 4 else K.unsqueeze(0)

Y_framework_single = F.conv2d(X_4d, K_single_out_4d)
Y_custom_single = corr2d_multi_in(X, K[0] if K.ndim == 4 else K)

print("--- 1. 多输入单输出验证 ---")
print("框架计算结果:\n", Y_framework_single.squeeze())
print("自定义计算结果:\n", Y_custom_single)
assert torch.allclose(Y_framework_single.squeeze(), Y_custom_single)
print("验证通过！误差小于容差范围。")

# 3. 验证：多输入多输出
# 此时 K 已经是一个 4D 权重张量，形状为 (out_channels=3, in_channels=2, kH=2, kW=2)
K_multi_out_4d = K if K.ndim == 4 else torch.stack((K, K + 1, K + 2), 0)

Y_framework_multi = F.conv2d(X_4d, K_multi_out_4d)
Y_custom_multi = corr2d_multi_in_out(X, K_multi_out_4d)

print("\n--- 2. 多输入多输出验证 ---")
print("框架计算结果:\n", Y_framework_multi.squeeze())
print("自定义计算结果:\n", Y_custom_multi)
assert torch.allclose(Y_framework_multi.squeeze(), Y_custom_multi)
print("验证通过！误差小于容差范围。")

# 4. 验证：使用面向对象 API (nn.Conv2d)
conv2d = nn.Conv2d(in_channels=2, out_channels=3, kernel_size=2, bias=False)
conv2d.weight.data = K_multi_out_4d  # 手动覆盖其权重以便与自定义示例完全一致
Y_nn_multi = conv2d(X_4d)
print("\n--- 3. nn.Conv2d 模块化验证 ---")
print("nn.Conv2d 输出形状:", Y_nn_multi.shape)
assert torch.allclose(Y_nn_multi.squeeze(), Y_custom_multi)
print("nn.Conv2d 验证通过！")

# 3. 1X1卷积（使用自定义）

In [ ]:
# 1×1卷积的多输入、多输出通道运算
def corr2d_multi_in_out_1x1(X,K):
    c_i, h, w = X.shape # 输入的通道数、宽、高
    c_o = K.shape[0]    # 输出的通道数
    X = X.reshape((c_i, h * w)) # 拉平操作，每一行表示一个通道 the 特征
    K = K.reshape((c_o,c_i)) 
    Y = torch.matmul(K,X) 
    return Y.reshape((c_o, h, w))

X = torch.normal(0,1,(3,3,3))   # norm函数生成0到1之间的(3,3,3)矩阵 
K = torch.normal(0,1,(2,3,1,1)) # 输出通道是2，输入通道是3，核是1X1

Y1 = corr2d_multi_in_out_1x1(X,K)
Y2 = corr2d_multi_in_out(X,K)
assert float(torch.abs(Y1-Y2).sum()) < 1e-6
print(float(torch.abs(Y1-Y2).sum()))

# 4. 1X1卷积（使用框架与验证）

In [ ]:
# 使用框架实现 1×1 卷积，并与自定义的矩阵乘法进行比对验证
import torch.nn.functional as F

# 1. 定义数据，使用与自定义一模一样的输入与核
X_1x1 = torch.normal(0, 1, (3, 3, 3))   # 3通道, H=3, W=3
K_1x1 = torch.normal(0, 1, (2, 3, 1, 1)) # 输出通道为2, 输入通道为3, 1x1核

# 2. 使用自定义的 1×1 矩阵相乘方法
Y_custom_1x1 = corr2d_multi_in_out_1x1(X_1x1, K_1x1)

# 3. 使用框架的二维卷积实现（F.conv2d）
X_1x1_4d = X_1x1.unsqueeze(0) # 升维为 4D 形状: (1, 3, 3, 3)
Y_framework_1x1 = F.conv2d(X_1x1_4d, K_1x1)

# 4. 使用 nn.Conv2d 实现
conv2d_1x1 = nn.Conv2d(in_channels=3, out_channels=2, kernel_size=1, bias=False)
conv2d_1x1.weight.data = K_1x1
Y_nn_1x1 = conv2d_1x1(X_1x1_4d)

print("--- 1x1 卷积验证 ---")
print("自定义 1x1 输出形状:", Y_custom_1x1.shape)
print("框架 F.conv2d 输出形状:", Y_framework_1x1.squeeze().shape)
print("框架 nn.Conv2d 输出形状:", Y_nn_1x1.squeeze().shape)

# 验证输出数值的等价性
assert torch.allclose(Y_custom_1x1, Y_framework_1x1.squeeze(), atol=1e-6)
assert torch.allclose(Y_custom_1x1, Y_nn_1x1.squeeze(), atol=1e-6)
print("1x1 卷积所有框架方法验证通过！与自定义矩阵乘法完全等价。\n")

# 原有的 padding 和 stride 框架函数调用演示（保持文档完整性）
def comp_conv2d(conv2d, X):
    X = X.reshape((1,1)+X.shape) # 升维成 4D
    Y = conv2d(X)
    return Y.reshape(Y.shape[2:]) # 降维回 2D

X_pad_stride = torch.rand(size=(8,8))
conv2d_p1_s2 = nn.Conv2d(1, 1, kernel_size=3, padding=1, stride=2)  # 先输入, 后输出 (in_channels=1, out_channels=1)
print("填充与步幅样例形状1:", comp_conv2d(conv2d_p1_s2, X_pad_stride).shape) 

conv2d_p01_s34 = nn.Conv2d(1, 1, kernel_size=(3,5), padding=(0,1), stride=(3,4))
print("填充与步幅样例形状2:", comp_conv2d(conv2d_p01_s34, X_pad_stride).shape)

# 5. 二维卷积层

以下是二维卷积层正向传播与输入输出形状关系的计算示意图：

![correlation](https://zh-v2.d2l.ai/_images/correlation.svg)